# Descarga de datos

Preparación de la categoría `Office_Products` del dataset Amazon Reviews 2023 (McAuley-Lab). El notebook tiene tres secciones.

1. **Descarga.** Se descargan las reseñas (`Office_Products.jsonl.gz`) y los metadatos de productos (`meta_Office_Products.jsonl.gz`) en formato JSON Lines comprimido.
2. **Preprocesamiento.** Se asigna el tipo de cada columna y se ordenan las reseñas por `timestamp` mediante ordenamiento externo.
3. **Exportación.** Se escriben `office_products.parquet` y `meta_office_products.parquet`.

Todos los archivos se guardan en `data/raw/`, carpeta excluida del control de versiones. Los `.jsonl.gz` se conservan después de la exportación.

## 1. Descarga

Cada archivo se descarga por bloques y se guarda en disco. Las variables `ruta_reviews` y `ruta_meta` guardan la ruta de cada archivo descargado, de modo que ningún archivo completo se carga en memoria.

In [1]:
import gzip
import heapq
import json
import shutil
import urllib.request
from pathlib import Path

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

In [2]:
CATEGORIA = "Office_Products"
# El host datarepo.eng.ucsd.edu indicado en Hugging Face responde 404; se usa el host del laboratorio
BASE_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"

URLS = {
    "reviews": f"{BASE_URL}/review_categories/{CATEGORIA}.jsonl.gz",
    "meta": f"{BASE_URL}/meta_categories/meta_{CATEGORIA}.jsonl.gz",
}

In [3]:
# Raiz del repositorio, tanto si el notebook se ejecuta desde notebooks/ como desde la raiz
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
DIR_CORRIDAS = RAW_DIR / "_corridas"  # corridas temporales del ordenamiento externo

ARCHIVOS = {clave: RAW_DIR / url.rsplit("/", 1)[-1] for clave, url in URLS.items()}
PARQUETS = {clave: ruta.with_name(ruta.name.removesuffix(".jsonl.gz").lower() + ".parquet") for clave, ruta in ARCHIVOS.items()}
ARCHIVOS, PARQUETS

({'reviews': WindowsPath('d:/projects/alg-final-project/data/raw/Office_Products.jsonl.gz'),
  'meta': WindowsPath('d:/projects/alg-final-project/data/raw/meta_Office_Products.jsonl.gz')},
 {'reviews': WindowsPath('d:/projects/alg-final-project/data/raw/office_products.parquet'),
  'meta': WindowsPath('d:/projects/alg-final-project/data/raw/meta_office_products.parquet')})

In [4]:
CHUNK = 8 * 1024 * 1024  # 8 MB por lectura


def descargar(url: str, destino: Path) -> Path:
    """Descarga url en destino por bloques y devuelve la ruta. Omite la descarga si el archivo ya existe."""
    if destino.exists():
        print(f"{destino.name} ya existe ({destino.stat().st_size / 1e6:,.0f} MB), se omite")
        return destino

    # Se escribe en un archivo temporal para no dejar un archivo incompleto con el nombre final
    temporal = destino.with_name(destino.name + ".part")
    with urllib.request.urlopen(url) as respuesta, open(temporal, "wb") as salida:
        total = int(respuesta.headers.get("Content-Length", 0))
        descargado = 0
        while bloque := respuesta.read(CHUNK):
            salida.write(bloque)
            descargado += len(bloque)
            print(f"\r{destino.name}: {descargado / 1e6:,.0f} / {total / 1e6:,.0f} MB", end="")
    print()
    temporal.rename(destino)
    return destino

In [5]:
ruta_reviews = descargar(URLS["reviews"], ARCHIVOS["reviews"])
ruta_meta = descargar(URLS["meta"], ARCHIVOS["meta"])

Office_Products.jsonl.gz: 1,617 / 1,617 MB
meta_Office_Products.jsonl.gz: 529 / 529 MB


## 2. Preprocesamiento

### 2.1 Asignación de tipos

Cada archivo se lee línea por línea y se agrupa en lotes de `LOTE` filas. Cada lote se convierte a una tabla Arrow con un esquema explícito, para que todos los lotes compartan los mismos tipos. No se limpian valores, porque la limpieza y la normalización corresponden a la etapa 1 del pipeline.

- Los campos anidados de estructura variable (`images`, `videos`, `details`, `author`) se guardan como texto JSON.
- `price` se guarda como texto porque el archivo original mezcla números, valores nulos y textos como `"from 14.99"` o `"—"`.
- `timestamp` se conserva como entero en milisegundos Unix.

In [6]:
ESQUEMA_REVIEWS = pa.schema([
    ("rating", pa.float64()),
    ("title", pa.string()),
    ("text", pa.string()),
    ("images", pa.string()),
    ("asin", pa.string()),
    ("parent_asin", pa.string()),
    ("user_id", pa.string()),
    ("timestamp", pa.int64()),
    ("helpful_vote", pa.int64()),
    ("verified_purchase", pa.bool_()),
])

ESQUEMA_META = pa.schema([
    ("main_category", pa.string()),
    ("title", pa.string()),
    ("average_rating", pa.float64()),
    ("rating_number", pa.int64()),
    ("features", pa.list_(pa.string())),
    ("description", pa.list_(pa.string())),
    ("price", pa.string()),
    ("images", pa.string()),
    ("videos", pa.string()),
    ("store", pa.string()),
    ("categories", pa.list_(pa.string())),
    ("details", pa.string()),
    ("parent_asin", pa.string()),
    ("bought_together", pa.list_(pa.string())),
    ("subtitle", pa.string()),
    ("author", pa.string()),
])

In [7]:
LOTE = 200_000  # filas por lote


def a_texto(valor):
    """Convierte listas y diccionarios a texto JSON y numeros a str. Conserva None y los textos."""
    if valor is None or isinstance(valor, str):
        return valor
    if isinstance(valor, (dict, list)):
        return json.dumps(valor, ensure_ascii=False)
    return str(valor)


def leer_lotes(origen: Path, esquema: pa.Schema):
    """Lee un JSON Lines comprimido y genera tablas Arrow de LOTE filas con los tipos del esquema."""
    columnas_texto = [campo.name for campo in esquema if campo.type == pa.string()]
    with gzip.open(origen, "rt", encoding="utf-8") as entrada:
        lote = []
        for linea in entrada:
            registro = json.loads(linea)
            for columna in columnas_texto:
                if columna in registro:
                    registro[columna] = a_texto(registro[columna])
            lote.append(registro)
            if len(lote) == LOTE:
                yield pa.Table.from_pylist(lote, schema=esquema)
                lote = []
        if lote:
            yield pa.Table.from_pylist(lote, schema=esquema)

### 2.2 Ordenamiento externo de las reseñas

Las reseñas se ordenan por `timestamp` con *external merge sort*, porque el archivo completo no se carga en memoria. Sea $n$ el número de filas, $L$ el tamaño de lote, $k = \lceil n / L \rceil$ el número de corridas y $b$ el tamaño de bloque de lectura durante la mezcla.

1. **Generación de corridas.** Cada lote de $L$ filas se ordena en memoria por `timestamp` y se escribe como una corrida temporal en `data/raw/_corridas/`. Costo $O(n \log L)$ y memoria $O(L)$.
2. **Mezcla de $k$ vías.** Se abre un iterador por corrida y un heap mínimo de tamaño $k$ guarda la fila actual de cada una. En cada paso se extrae la fila con menor `timestamp` y se reemplaza por la siguiente fila de su misma corrida. Costo $O(n \log k)$ y memoria $O(k \cdot b)$.

El costo total es $O(n \log L + n \log k) = O(n \log n)$, porque $L \cdot k \approx n$. El ordenamiento es estable. El ordenamiento de Arrow dentro de cada corrida es estable, y en el heap los empates de `timestamp` se resuelven por el índice de la corrida, de modo que las filas con igual `timestamp` conservan su orden original.

Los metadatos no se ordenan, porque son un catálogo que las etapas posteriores consultan por clave.

In [8]:
BLOQUE_MEZCLA = 5_000  # filas leidas por corrida en cada acceso durante la mezcla


def crear_corridas(lotes, carpeta: Path, clave: str) -> list[Path]:
    """Ordena cada lote por clave y lo escribe como corrida temporal. Devuelve las rutas de las corridas."""
    carpeta.mkdir(parents=True, exist_ok=True)
    corridas = []
    filas = 0
    for i, lote in enumerate(lotes):
        ruta = carpeta / f"corrida_{i:03d}.parquet"
        pq.write_table(lote.sort_by(clave), ruta)
        corridas.append(ruta)
        filas += lote.num_rows
        print(f"\r{len(corridas)} corridas, {filas:,} filas", end="")
    print()
    return corridas


def filas_de_corrida(ruta: Path):
    """Genera las filas de una corrida leyendo bloques de BLOQUE_MEZCLA filas."""
    for bloque in pq.ParquetFile(ruta).iter_batches(batch_size=BLOQUE_MEZCLA):
        yield from bloque.to_pylist()


def mezclar_corridas(corridas: list[Path], clave: str, esquema: pa.Schema):
    """Mezcla k corridas ordenadas con un heap minimo y genera tablas Arrow de LOTE filas en orden global."""
    iteradores = [filas_de_corrida(ruta) for ruta in corridas]

    # El heap guarda (clave, indice de corrida, fila); el indice resuelve empates y evita comparar filas
    heap = []
    for i, iterador in enumerate(iteradores):
        fila = next(iterador, None)
        if fila is not None:
            heap.append((fila[clave], i, fila))
    heapq.heapify(heap)

    lote = []
    while heap:
        _, i, fila = heap[0]
        lote.append(fila)
        siguiente = next(iteradores[i], None)
        if siguiente is None:
            heapq.heappop(heap)
        else:
            heapq.heapreplace(heap, (siguiente[clave], i, siguiente))
        if len(lote) == LOTE:
            yield pa.Table.from_pylist(lote, schema=esquema)
            lote = []
    if lote:
        yield pa.Table.from_pylist(lote, schema=esquema)

Las corridas se generan en esta celda. La mezcla y la lectura de los metadatos son generadores y se ejecutan durante la exportación, a medida que se escribe cada lote.

In [9]:
if PARQUETS["reviews"].exists():
    print(f"{PARQUETS['reviews'].name} ya existe, se omite el ordenamiento")
    corridas = []
else:
    corridas = crear_corridas(leer_lotes(ruta_reviews, ESQUEMA_REVIEWS), DIR_CORRIDAS, "timestamp")

lotes_reviews = mezclar_corridas(corridas, "timestamp", ESQUEMA_REVIEWS)
lotes_meta = leer_lotes(ruta_meta, ESQUEMA_META)

65 corridas, 12,845,712 filas


## 3. Exportación

Cada secuencia de lotes se escribe en un Parquet con compresión zstd. La escritura usa un archivo `.part` que se renombra al terminar, para no dejar un archivo incompleto con el nombre final. Después de exportar las reseñas se eliminan las corridas temporales.

In [10]:
def exportar(lotes, destino: Path, esquema: pa.Schema) -> Path:
    """Escribe los lotes en destino como Parquet y devuelve la ruta. Omite la escritura si el destino ya existe."""
    if destino.exists():
        print(f"{destino.name} ya existe ({destino.stat().st_size / 1e6:,.0f} MB), se omite")
        return destino

    temporal = destino.with_name(destino.name + ".part")
    filas = 0
    with pq.ParquetWriter(temporal, esquema, compression="zstd") as escritor:
        for lote in lotes:
            escritor.write_table(lote)
            filas += lote.num_rows
            print(f"\r{destino.name}: {filas:,} filas", end="")
    print()
    temporal.rename(destino)
    return destino

In [11]:
parquet_reviews = exportar(lotes_reviews, PARQUETS["reviews"], ESQUEMA_REVIEWS)
shutil.rmtree(DIR_CORRIDAS, ignore_errors=True)
parquet_meta = exportar(lotes_meta, PARQUETS["meta"], ESQUEMA_META)

office_products.parquet: 12,845,712 filas
meta_office_products.parquet: 710,503 filas


### Verificación

Número de filas, columnas y tamaño de cada Parquet, comprobación del orden de las reseñas y primeras filas de cada archivo. La comprobación del orden lee solo la columna `timestamp`, y las muestras leen solo el primer lote.

In [12]:
for ruta in (parquet_reviews, parquet_meta):
    metadatos = pq.read_metadata(ruta)
    print(f"{ruta.name}: {metadatos.num_rows:,} filas, {metadatos.num_columns} columnas, {ruta.stat().st_size / 1e6:,.0f} MB")

timestamps = pq.read_table(parquet_reviews, columns=["timestamp"])["timestamp"]
print("Reseñas ordenadas por timestamp:", pc.all(pc.greater_equal(timestamps[1:], timestamps[:-1])).as_py())

office_products.parquet: 12,845,712 filas, 10 columnas, 1,495 MB
meta_office_products.parquet: 710,503 filas, 16 columnas, 505 MB
Reseñas ordenadas por timestamp: True


In [13]:
reviews_muestra = next(pq.ParquetFile(parquet_reviews).iter_batches(batch_size=5)).to_pandas()
reviews_muestra

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,"The ""Starters Kit"" to collecting the ""Stones""",Anyone who is just begining to appreciate The ...,[],B000000W5N,B000000W5N,AHTNR4HTDYAQQ2MRCGKJGPNCEKUQ,908165260000,4,False
1,5.0,"Definitive Stones, definitive Rock; probably t...",Every song on this album is great. There are ...,[],B000000W5N,B000000W5N,AGUGSJHL7E4APYHVX3PG4PZEVONQ,910174425000,12,False
2,5.0,Ultimate depressing album.,"As a young Beatles fan, I was quickly drawn to...",[],B000000W5N,B000000W5N,AGZRB4IT4WMNK4YSTOHMABCPPL2A,914740385000,12,False
3,1.0,This book is too simplistic and contains erron...,This book does not adequately differentiate be...,[],1891112511,1891112511,AH6DVUM45UBMKC4XZY7H4O3YQE7A,928882812000,6,False
4,3.0,Mediocre improvements,I have a US Robotics Palm Pro (we go back a wa...,[],B00000J4FS,B00000J4FS,AHYUBHKG532QOWIMEA6J6PIIKJGA,930053323000,0,False


In [14]:
meta_muestra = next(pq.ParquetFile(parquet_meta).iter_batches(batch_size=5)).to_pandas()
meta_muestra

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Office Products,Alliance Rubber 07706 Non-Latex Brites File Ba...,4.5,665,[REUSABLE: These colored rubber bands are stre...,[Alliance Rubber Brites File Bands are durable...,2.68,"[{""thumb"": ""https://m.media-amazon.com/images/...","[{""title"": ""AMUU rubber bands Large 50 Pack 5 ...",Alliance,"[Office Products, Office & School Supplies, Ta...","{""Manufacturer"": ""Alliance Rubber Company Inc....",B001S28Q4Q,None,NaN,NaN
1,Office Products,"10 Set Mini Office Stapler with 10# Staples, S...",3.3,186,[Made of high quality metal & plastic for dura...,"[Whether you're a teacher, office worker or st...",13.99,"[{""thumb"": ""https://m.media-amazon.com/images/...","[{""title"": ""Small Mini stapler "", ""url"": ""http...",Chneeu,"[Office Products, Office & School Supplies, St...","{""Manufacturer"": ""Chnee"", ""Brand"": ""Chneeu"", ""...",B07RPQVH21,None,NaN,NaN
2,Office Products,Creanoso Humorous Sayings Bookish Bookmarks fo...,4.6,100,[Creanoso Bookish Humor and Sayings Bookmarks ...,[],9.99,"[{""thumb"": ""https://m.media-amazon.com/images/...","[{""title"": ""Cool Space Starry Night Bookmarks""...",Creanoso,"[Office Products, Office & School Supplies, Bo...","{""Manufacturer"": ""Creanoso"", ""Brand"": ""Creanos...",B07J58B87Q,None,NaN,NaN
3,All Electronics,PenPower WorldPen Scan Go | OCR Reading Pen wi...,3.9,296,[TEXT TO SPEECH - Scan printed or digital text...,[],149.0,"[{""thumb"": ""https://m.media-amazon.com/images/...","[{""title"": ""Easy to use"", ""url"": ""https://www....",PenPower,"[Office Products, Office Electronics, Scanners...","{""Product Dimensions"": ""0.56\""D x 1.24\""W x 5....",B098NQ685V,None,NaN,NaN
4,Amazon Home,"Assorted Best Selling Birthday Cards, Set of 8...",4.0,11,[],[],NaN,"[{""thumb"": ""https://m.media-amazon.com/images/...",[],Apartment 2 Cards,"[Office Products, Office & School Supplies, Pa...","{""Product Dimensions"": ""5.5 x 4.25 x 0.13 inch...",B00DBSNIC0,None,NaN,NaN
